# Backtest Framework Comparsion

This notebook compares the same 50/200 SMA crossover portfolio across data vendors and backtesting frameworks. The point is not to make SMA crossover a platform abstraction; it is a compact test case for whether Quant Orchestrator can hold the strategy constant while changing data providers and execution engines.

Data and SMA features come from Quant Warehouse. The backtesting engines receive prepared in-memory frames and do not compute their own indicators. The notebook controls date coverage by clipping every symbol/provider pair to one shared global calendar. Price adjustment is handled upstream by Quant Warehouse, which requests dividend-and-split adjusted daily equity OHLC from OpenBB.

In [1]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import sys
import warnings

warnings.filterwarnings("ignore", message="Jupyter Notebook detected.*", category=UserWarning)
warnings.filterwarnings("ignore", message="DataFrameGroupBy.apply operated on the grouping columns.*", category=FutureWarning)
warnings.filterwarnings("ignore", message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated.*", category=FutureWarning)

import pandas as pd
from IPython.display import Markdown, display

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "quant_orchestrator").exists():
            return candidate
    raise RuntimeError("Could not find quant-orchestrator repo root")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quant_warehouse import Warehouse
from quant_orchestrator.pipeline import PipelineContext
from quant_orchestrator.backtests.research import (
    FrameworkRun,
    _coerce_framework_run,
    _framework_comparison_rows,
    _run_nautilus_isolated,
    build_sma_frame,
    run_backtesting_py,
    run_zipline,
)
from quant_orchestrator.platforms.backtesting_frameworks.reporting import build_common_summary
from quant_orchestrator.platforms.backtesting_frameworks.shared import (
    MAG7_SYMBOLS,
    OHLCV_COLUMNS,
    combine_equity_curves,
    equal_notional_capital,
)

## Setup

The experiment is a single MAG7 portfolio, not seven independent headline backtests. Each symbol gets an equal capital sleeve and the portfolio equity curve is the sum of those sleeves. We then compare the portfolio-level result for each provider/framework pair.

To keep the comparison fair, all providers use the same global symbol/provider calendar after loading. Quant Warehouse is expected to provide adjusted OHLC, so this notebook audits the basis but does not apply notebook-side price adjustments.

In [2]:
SYMBOLS = MAG7_SYMBOLS
PROVIDERS = ("yfinance", "fmp")
FRAMEWORKS = ("backtesting.py", "zipline", "nautilus")
START = "2020-01-01"
END = None
FAST_WINDOW = 50
SLOW_WINDOW = 200
CAPITAL_BASE = 100_000.0
PRICE_BASIS = "warehouse_splits_and_dividends_adjusted_ohlc"
DATE_ALIGNMENT = "global_provider_symbol_intersection"

CONFIG = pd.DataFrame(
    [
        {
            "symbols": ", ".join(SYMBOLS),
            "providers": ", ".join(PROVIDERS),
            "frameworks": ", ".join(FRAMEWORKS),
            "start": START,
            "end": END or "latest available",
            "fast_window": FAST_WINDOW,
            "slow_window": SLOW_WINDOW,
            "capital_base": CAPITAL_BASE,
            "capital_per_symbol": equal_notional_capital(CAPITAL_BASE, len(SYMBOLS)),
            "price_basis": PRICE_BASIS,
            "date_alignment": DATE_ALIGNMENT,
        }
    ]
)
display(CONFIG.T.rename(columns={0: "value"}))

pipeline_context = PipelineContext(metadata={"notebook": "sample_strategy_comparsion"})
_ = pipeline_context.set("config", CONFIG)

,value
symbols,"AAPL, AMZN, GOOGL, META, MSFT, NVDA, TSLA"
providers,"yfinance, fmp"
frameworks,"backtesting.py, zipline, nautilus"
start,2020-01-01
end,latest available
fast_window,50
slow_window,200
capital_base,100000.0
capital_per_symbol,14285.714286
price_basis,warehouse_splits_and_dividends_adjusted_ohlc


In [3]:
def _normalize_index(frame: pd.DataFrame) -> pd.DataFrame:
    normalized = frame.rename(columns=str.lower).copy()
    normalized.index = pd.DatetimeIndex(normalized.index)
    if normalized.index.tz is not None:
        normalized.index = normalized.index.tz_convert(None)
    return normalized.sort_index()


def standardize_price_basis(frame: pd.DataFrame, *, provider: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    standardized = _normalize_index(frame)
    raw_close = standardized["close"].astype(float).copy()
    standardized = standardized.loc[:, list(OHLCV_COLUMNS)].copy()
    standardized["volume"] = standardized["volume"].fillna(0.0)
    audit = pd.DataFrame(
        [
            {
                "provider": provider,
                "raw_start_close": float(raw_close.iloc[0]),
                "standardized_start_close": float(standardized["close"].iloc[0]),
                "raw_end_close": float(raw_close.iloc[-1]),
                "standardized_end_close": float(standardized["close"].iloc[-1]),
                "min_adjustment_factor": 1.0,
                "max_adjustment_factor": 1.0,
                "basis_method": "warehouse_adjusted_ohlc_no_notebook_rewrite",
            }
        ]
    )
    return standardized, audit


def load_all_provider_prices() -> dict[str, dict[str, pd.DataFrame]]:
    warehouse = Warehouse()
    return {
        provider: {
            symbol: warehouse.read_prices(symbol, provider=provider, start=START, end=END)
            for symbol in SYMBOLS
        }
        for provider in PROVIDERS
    }


def prepare_comparison_frames(
    raw_frames: dict[str, dict[str, pd.DataFrame]],
) -> tuple[dict[str, dict[str, pd.DataFrame]], pd.DataFrame, pd.DataFrame]:
    standardized: dict[str, dict[str, pd.DataFrame]] = {provider: {} for provider in PROVIDERS}
    basis_audits = []

    for provider in PROVIDERS:
        for symbol in SYMBOLS:
            frame, audit = standardize_price_basis(raw_frames[provider][symbol], provider=provider)
            standardized[provider][symbol] = frame
            audit.insert(0, "symbol", symbol)
            basis_audits.append(audit)

    global_index = None
    for symbol in SYMBOLS:
        for provider in PROVIDERS:
            provider_index = standardized[provider][symbol].index
            global_index = provider_index if global_index is None else global_index.intersection(provider_index)
    global_index = pd.DatetimeIndex(global_index).sort_values()
    if global_index.empty:
        raise ValueError("No common dates across all symbols and providers")

    aligned: dict[str, dict[str, pd.DataFrame]] = {provider: {} for provider in PROVIDERS}
    date_rows = []
    for symbol in SYMBOLS:
        for provider in PROVIDERS:
            before = standardized[provider][symbol]
            after = before.loc[global_index].copy()
            aligned[provider][symbol] = after
            date_rows.append(
                {
                    "symbol": symbol,
                    "provider": provider,
                    "raw_start": before.index.min().date().isoformat(),
                    "raw_end": before.index.max().date().isoformat(),
                    "raw_rows": len(before),
                    "aligned_start": after.index.min().date().isoformat(),
                    "aligned_end": after.index.max().date().isoformat(),
                    "aligned_rows": len(after),
                    "dropped_rows": len(before) - len(after),
                }
            )

    basis_audit = pd.concat(basis_audits, ignore_index=True)
    first_close = (
        pd.concat(
            [
                aligned[provider][symbol]["close"].rename((symbol, provider))
                for symbol in SYMBOLS
                for provider in PROVIDERS
            ],
            axis=1,
        )
        .iloc[0]
        .rename("first_aligned_standardized_close")
        .reset_index()
        .rename(columns={"level_0": "symbol", "level_1": "provider"})
    )
    basis_audit = basis_audit.merge(first_close, on=["symbol", "provider"], how="left")
    return aligned, pd.DataFrame(date_rows), basis_audit

def run_symbol_sleeve(
    framework: str,
    *,
    symbol: str,
    prices: pd.DataFrame,
    capital_base: float,
) -> FrameworkRun:
    if framework == "nautilus":
        return _run_nautilus_isolated(
            prices,
            symbol=symbol,
            fast_window=FAST_WINDOW,
            slow_window=SLOW_WINDOW,
            capital_base=capital_base,
        )

    signal_frame = build_sma_frame(prices, fast_window=FAST_WINDOW, slow_window=SLOW_WINDOW)
    if framework == "backtesting.py":
        return _coerce_framework_run(run_backtesting_py(signal_frame, symbol=symbol, capital_base=capital_base))
    if framework == "zipline":
        return _coerce_framework_run(run_zipline(signal_frame, symbol=symbol, capital_base=capital_base))
    raise ValueError(f"Unsupported framework: {framework}")


def run_mag7_portfolio(
    *,
    provider: str,
    framework: str,
    price_frames: dict[str, pd.DataFrame],
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, FrameworkRun]]:
    capital_per_symbol = equal_notional_capital(CAPITAL_BASE, len(price_frames))
    started = perf_counter()
    sleeves = []
    runs = {}
    symbol_rows = []
    trades = 0
    native_elapsed_seconds = 0.0

    for symbol, prices in price_frames.items():
        run = run_symbol_sleeve(
            framework,
            symbol=symbol,
            prices=prices,
            capital_base=capital_per_symbol,
        )
        runs[symbol] = run
        sleeves.append(run.equity.rename(symbol))
        row = run.summary.copy()
        row["provider"] = provider
        row["framework"] = framework
        symbol_rows.append(row)
        trades += int(run.summary["trades"].iloc[0])
        native_elapsed_seconds += float(run.summary["elapsed_seconds"].iloc[0])

    portfolio_equity = combine_equity_curves(sleeves)
    wall_clock_seconds = perf_counter() - started
    portfolio_summary = build_common_summary(
        framework=framework,
        symbol="MAG7",
        equity=portfolio_equity,
        elapsed_seconds=wall_clock_seconds,
        bars=len(portfolio_equity),
        trades=trades,
    )
    portfolio_summary["provider"] = provider
    portfolio_summary["fast_window"] = FAST_WINDOW
    portfolio_summary["slow_window"] = SLOW_WINDOW
    portfolio_summary["capital_base"] = CAPITAL_BASE
    portfolio_summary["capital_per_symbol"] = capital_per_symbol
    portfolio_summary["price_basis"] = PRICE_BASIS
    portfolio_summary["date_alignment"] = DATE_ALIGNMENT
    portfolio_summary["native_elapsed_seconds"] = round(native_elapsed_seconds, 4)
    portfolio_summary["wall_clock_seconds"] = round(wall_clock_seconds, 4)
    portfolio_summary["performance_score"] = (
        portfolio_summary["total_return"] + portfolio_summary["max_drawdown"]
    )
    return portfolio_summary, pd.concat(symbol_rows, ignore_index=True), runs

## Run Jobs

Each row below is a Quant Orchestrator-style job: one data provider, one backtesting framework, one equal-notional MAG7 portfolio strategy. The first two outputs are audit tables showing the price-basis standardization and common-date alignment applied before any backtest runs.

In [4]:
raw_price_frames = load_all_provider_prices()
price_frames_by_provider, date_alignment_audit, price_basis_audit = prepare_comparison_frames(raw_price_frames)

display(date_alignment_audit)
display(price_basis_audit)
_ = pipeline_context.update({"raw_price_frames": raw_price_frames, "price_frames_by_provider": price_frames_by_provider, "date_alignment_audit": date_alignment_audit, "price_basis_audit": price_basis_audit})

portfolio_rows = []
symbol_rows = []
runs: dict[str, dict[str, dict[str, FrameworkRun]]] = {}

for provider in PROVIDERS:
    runs[provider] = {}
    for framework in FRAMEWORKS:
        portfolio_summary, symbol_summary, framework_runs = run_mag7_portfolio(
            provider=provider,
            framework=framework,
            price_frames=price_frames_by_provider[provider],
        )
        portfolio_rows.append(portfolio_summary)
        symbol_rows.append(symbol_summary)
        runs[provider][framework] = framework_runs

comparison = (
    pd.concat(portfolio_rows, ignore_index=True)
    .sort_values(["provider", "framework"])
    .reset_index(drop=True)
)
slowest_wall_clock_seconds = float(comparison["wall_clock_seconds"].max())
comparison["speedup_vs_slowest"] = slowest_wall_clock_seconds / comparison["wall_clock_seconds"]
symbol_detail = pd.concat(symbol_rows, ignore_index=True).sort_values(["provider", "framework", "symbol"]).reset_index(drop=True)
comparison

,symbol,provider,raw_start,raw_end,raw_rows,aligned_start,aligned_end,aligned_rows,dropped_rows
0,AAPL,yfinance,2020-01-02,2026-06-26,1629,2020-01-02,2026-06-24,1627,2
1,AAPL,fmp,2020-01-02,2026-06-24,1627,2020-01-02,2026-06-24,1627,0
2,AMZN,yfinance,2020-01-02,2026-06-26,1629,2020-01-02,2026-06-24,1627,2
3,AMZN,fmp,2020-01-02,2026-06-24,1627,2020-01-02,2026-06-24,1627,0
4,GOOGL,yfinance,2020-01-02,2026-06-26,1629,2020-01-02,2026-06-24,1627,2
5,GOOGL,fmp,2020-01-02,2026-06-24,1627,2020-01-02,2026-06-24,1627,0
6,META,yfinance,2020-01-02,2026-06-26,1629,2020-01-02,2026-06-24,1627,2
7,META,fmp,2020-01-02,2026-06-24,1627,2020-01-02,2026-06-24,1627,0
8,MSFT,yfinance,2020-01-02,2026-06-26,1629,2020-01-02,2026-06-24,1627,2
9,MSFT,fmp,2020-01-02,2026-06-24,1627,2020-01-02,2026-06-24,1627,0


,symbol,provider,raw_start_close,standardized_start_close,raw_end_close,standardized_end_close,min_adjustment_factor,max_adjustment_factor,basis_method,first_aligned_standardized_close
0,AAPL,yfinance,72.333870,72.333870,283.779999,283.779999,1.0,1.0,warehouse_adjusted_ohlc_no_notebook_rewrite,72.333870
1,AMZN,yfinance,94.900497,94.900497,232.690002,232.690002,1.0,1.0,warehouse_adjusted_ohlc_no_notebook_rewrite,94.900497
2,GOOGL,yfinance,67.832512,67.832512,337.390015,337.390015,1.0,1.0,warehouse_adjusted_ohlc_no_notebook_rewrite,67.832512
3,META,yfinance,207.953857,207.953857,550.250000,550.250000,1.0,1.0,warehouse_adjusted_ohlc_no_notebook_rewrite,207.953857
4,MSFT,yfinance,151.829544,151.829544,372.970001,372.970001,1.0,1.0,warehouse_adjusted_ohlc_no_notebook_rewrite,151.829544
5,NVDA,yfinance,5.963803,5.963803,192.529999,192.529999,1.0,1.0,warehouse_adjusted_ohlc_no_notebook_rewrite,5.963803
6,TSLA,yfinance,28.684000,28.684000,379.709991,379.709991,1.0,1.0,warehouse_adjusted_ohlc_no_notebook_rewrite,28.684000
7,AAPL,fmp,72.330000,72.330000,293.080000,293.080000,1.0,1.0,warehouse_adjusted_ohlc_no_notebook_rewrite,72.330000
8,AMZN,fmp,94.900000,94.900000,234.270000,234.270000,1.0,1.0,warehouse_adjusted_ohlc_no_notebook_rewrite,94.900000
9,GOOGL,fmp,68.430000,68.430000,345.290000,345.290000,1.0,1.0,warehouse_adjusted_ohlc_no_notebook_rewrite,68.430000


Loading BokehJS ...

Backtest.run:   0%|          | 0/1626 [00:00<?, ?bar/s]

/home/jlee153232/PycharmProjects/quant-orchestrator/quant_orchestrator/backtests/research.py:353: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  ).run()


...

/home/jlee153232/miniconda3/lib/python3.13/site-packages/zipline/finance/metrics/tracker.py:127: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  registered.append(getattr(metric, hook))


/home/jlee153232/miniconda3/lib/python3.13/site-packages/zipline/finance/metrics/tracker.py:127: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  registered.append(getattr(metric, hook))


/home/jlee153232/miniconda3/lib/python3.13/site-packages/zipline/finance/metrics/tracker.py:127: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  registered.append(getattr(metric, hook))


/home/jlee153232/miniconda3/lib/python3.13/site-packages/zipline/finance/metrics/tracker.py:127: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  registered.append(getattr(metric, hook))


,framework,symbol,start,end,bars,trades,initial_equity,final_equity,total_return,annualized_return,...,fast_window,slow_window,capital_base,capital_per_symbol,price_basis,date_alignment,native_elapsed_seconds,wall_clock_seconds,performance_score,speedup_vs_slowest
0,backtesting.py,MAG7,2020-01-02,2026-06-24,1627,20,100000.0,187592.01,0.8759,0.1023,...,50,200,100000.0,14285.714286,warehouse_splits_and_dividends_adjusted_ohlc,global_provider_symbol_intersection,0.1801,0.2766,0.7305,18.496385
1,nautilus,MAG7,2020-01-02,2026-06-24,1627,45,100000.0,240630.39,1.4063,0.1457,...,50,200,100000.0,14285.714286,warehouse_splits_and_dividends_adjusted_ohlc,global_provider_symbol_intersection,0.5399,5.1161,1.0987,1.000000
2,zipline,MAG7,2020-01-02,2026-06-24,1627,60,100000.0,211225.83,1.1123,0.1228,...,50,200,100000.0,14285.714286,warehouse_splits_and_dividends_adjusted_ohlc,global_provider_symbol_intersection,1.9929,2.1982,0.8802,2.327404
3,backtesting.py,MAG7,2020-01-02,2026-06-24,1627,19,100000.0,188391.29,0.8839,0.1031,...,50,200,100000.0,14285.714286,warehouse_splits_and_dividends_adjusted_ohlc,global_provider_symbol_intersection,0.1774,0.4259,0.7394,12.012444
4,nautilus,MAG7,2020-01-02,2026-06-24,1627,43,100000.0,242248.77,1.4225,0.1469,...,50,200,100000.0,14285.714286,warehouse_splits_and_dividends_adjusted_ohlc,global_provider_symbol_intersection,0.5371,5.0068,1.1161,1.021830
5,zipline,MAG7,2020-01-02,2026-06-24,1627,58,100000.0,212774.17,1.1277,0.1241,...,50,200,100000.0,14285.714286,warehouse_splits_and_dividends_adjusted_ohlc,global_provider_symbol_intersection,2.1893,2.9078,0.8957,1.759440


## Portfolio Comparison

This table is the main result. Returns and drawdowns are portfolio-level values after combining the equal-notional symbol sleeves. The `start` and `end` columns should now match by framework across providers because every symbol/provider frame was clipped to the same global intersection before running.

In [5]:
comparison_display = comparison.loc[:, [
    "provider",
    "framework",
    "symbol",
    "start",
    "end",
    "trades",
    "final_equity",
    "total_return",
    "max_drawdown",
    "annualized_vol",
    "sharpe",
    "performance_score",
    "price_basis",
    "date_alignment",
    "wall_clock_seconds",
    "speedup_vs_slowest",
    "bars_per_second",
]].copy()
for column in ["total_return", "max_drawdown", "annualized_vol", "performance_score"]:
    comparison_display[column] = (comparison_display[column] * 100).round(2).astype(str) + "%"
for column in ["final_equity", "sharpe", "wall_clock_seconds", "speedup_vs_slowest", "bars_per_second"]:
    comparison_display[column] = comparison_display[column].round(2)
display(comparison_display)

,provider,framework,symbol,start,end,trades,final_equity,total_return,max_drawdown,annualized_vol,sharpe,performance_score,price_basis,date_alignment,wall_clock_seconds,speedup_vs_slowest,bars_per_second
0,fmp,backtesting.py,MAG7,2020-01-02,2026-06-24,20,187592.01,87.59%,-14.54%,12.08%,0.87,73.05%,warehouse_splits_and_dividends_adjusted_ohlc,global_provider_symbol_intersection,0.28,18.50,5882.28
1,fmp,nautilus,MAG7,2020-01-02,2026-06-24,45,240630.39,140.63%,-30.76%,20.51%,0.77,109.87%,warehouse_splits_and_dividends_adjusted_ohlc,global_provider_symbol_intersection,5.12,1.00,318.01
2,fmp,zipline,MAG7,2020-01-02,2026-06-24,60,211225.83,111.23%,-23.21%,19.62%,0.69,88.02%,warehouse_splits_and_dividends_adjusted_ohlc,global_provider_symbol_intersection,2.20,2.33,740.16
3,yfinance,backtesting.py,MAG7,2020-01-02,2026-06-24,19,188391.29,88.39%,-14.45%,12.09%,0.87,73.94%,warehouse_splits_and_dividends_adjusted_ohlc,global_provider_symbol_intersection,0.43,12.01,3819.77
4,yfinance,nautilus,MAG7,2020-01-02,2026-06-24,43,242248.77,142.25%,-30.64%,20.49%,0.77,111.61%,warehouse_splits_and_dividends_adjusted_ohlc,global_provider_symbol_intersection,5.01,1.02,324.96
5,yfinance,zipline,MAG7,2020-01-02,2026-06-24,58,212774.17,112.77%,-23.2%,19.61%,0.69,89.57%,warehouse_splits_and_dividends_adjusted_ohlc,global_provider_symbol_intersection,2.91,1.76,559.52


## Symbol Detail

The strategy is reported as a MAG7 portfolio, but the symbol-level sleeves are shown here to make the portfolio construction auditable.

In [6]:
symbol_detail_display = symbol_detail.loc[:, [
    "provider",
    "framework",
    "symbol",
    "trades",
    "final_equity",
    "total_return",
    "max_drawdown",
    "elapsed_seconds",
]].copy()
for column in ["total_return", "max_drawdown"]:
    symbol_detail_display[column] = (symbol_detail_display[column] * 100).round(2).astype(str) + "%"
for column in ["final_equity", "elapsed_seconds"]:
    symbol_detail_display[column] = symbol_detail_display[column].round(2)
display(symbol_detail_display)

,provider,framework,symbol,trades,final_equity,total_return,max_drawdown,elapsed_seconds
0,fmp,backtesting.py,AAPL,4,17114.48,19.8%,-21.1%,0.03
1,fmp,backtesting.py,AMZN,4,13496.13,-5.53%,-23.49%,0.03
2,fmp,backtesting.py,GOOGL,2,27047.55,89.33%,-15.11%,0.03
3,fmp,backtesting.py,META,3,21362.81,49.54%,-17.73%,0.03
4,fmp,backtesting.py,MSFT,5,17108.09,19.76%,-15.16%,0.03
5,fmp,backtesting.py,NVDA,2,77177.21,440.24%,-27.96%,0.03
6,fmp,backtesting.py,TSLA,0,14285.71,0.0%,0.0%,0.02
7,fmp,nautilus,AAPL,9,17027.75,19.19%,-20.93%,0.10
8,fmp,nautilus,AMZN,9,13313.72,-6.8%,-24.74%,0.09
9,fmp,nautilus,GOOGL,5,27184.83,90.29%,-15.0%,0.08


## Difference Attribution

This is a small two-way decomposition over the 2 providers x 3 frameworks matrix. It is not a statistical test; it is a practical magnitude check for whether the provider choice or framework choice moved a metric more in this fixed strategy example.

In [7]:
factor_report = pd.concat(
    [
        _framework_comparison_rows(comparison, metric="performance_score"),
        _framework_comparison_rows(comparison, metric="total_return"),
        _framework_comparison_rows(comparison, metric="max_drawdown"),
        _framework_comparison_rows(comparison, metric="wall_clock_seconds"),
        _framework_comparison_rows(comparison, metric="speedup_vs_slowest"),
    ],
    ignore_index=True,
)
factor_display = factor_report.copy()
for column in ["framework_share", "provider_share", "residual_share"]:
    factor_display[column] = (factor_display[column] * 100).round(2).astype(str) + "%"
for column in ["framework_ss", "provider_ss", "residual_ss", "provider_to_framework_ratio"]:
    factor_display[column] = factor_display[column].round(6)
display(factor_display)

,metric,framework_ss,provider_ss,residual_ss,framework_share,provider_share,residual_share,dominant_factor,provider_to_framework_ratio
0,performance_score,0.140191,0.000291,0.000020,99.78%,0.21%,0.01%,framework,0.002077
1,total_return,0.286673,0.000261,0.000020,99.9%,0.09%,0.01%,framework,0.000912
2,max_drawdown,0.026309,0.000001,0.000000,100.0%,0.0%,0.0%,framework,0.000031
3,wall_clock_seconds,22.217339,0.093650,0.175235,98.8%,0.42%,0.78%,framework,0.004215
4,speedup_vs_slowest,252.315773,8.236991,12.945281,92.26%,3.01%,4.73%,framework,0.032646


## Summaries

These summaries average over the other axis. They are useful for quick scanning, but the detailed comparison table above is the source of truth for this run.

In [8]:
framework_summary = comparison.groupby("framework").agg(
    mean_total_return=("total_return", "mean"),
    mean_max_drawdown=("max_drawdown", "mean"),
    mean_annualized_vol=("annualized_vol", "mean"),
    mean_sharpe=("sharpe", "mean"),
    mean_performance_score=("performance_score", "mean"),
    mean_wall_clock_seconds=("wall_clock_seconds", "mean"),
    mean_speedup_vs_slowest=("speedup_vs_slowest", "mean"),
    mean_bars_per_second=("bars_per_second", "mean"),
).sort_values("mean_performance_score", ascending=False)
provider_summary = comparison.groupby("provider").agg(
    mean_total_return=("total_return", "mean"),
    mean_max_drawdown=("max_drawdown", "mean"),
    mean_annualized_vol=("annualized_vol", "mean"),
    mean_sharpe=("sharpe", "mean"),
    mean_performance_score=("performance_score", "mean"),
    mean_wall_clock_seconds=("wall_clock_seconds", "mean"),
    mean_speedup_vs_slowest=("speedup_vs_slowest", "mean"),
    mean_bars_per_second=("bars_per_second", "mean"),
).sort_values("mean_performance_score", ascending=False)

display(framework_summary)
display(provider_summary)

,mean_total_return,mean_max_drawdown,mean_annualized_vol,mean_sharpe,mean_performance_score,mean_wall_clock_seconds,mean_speedup_vs_slowest,mean_bars_per_second
framework,,,,,,,,
nautilus,1.4144,-0.30700,0.20500,0.76925,1.10740,5.06145,1.010915,321.485
zipline,1.1200,-0.23205,0.19615,0.69195,0.88795,2.55300,2.043422,649.840
backtesting.py,0.8799,-0.14495,0.12085,0.86985,0.73495,0.35125,15.254414,4851.025


,mean_total_return,mean_max_drawdown,mean_annualized_vol,mean_sharpe,mean_performance_score,mean_wall_clock_seconds,mean_speedup_vs_slowest,mean_bars_per_second
provider,,,,,,,,
yfinance,1.1447,-0.227633,0.173967,0.779767,0.917067,2.780167,4.931238,1568.083333
fmp,1.1315,-0.228367,0.174033,0.774267,0.903133,2.530300,7.274596,2313.483333


## Written Analysis

The following cell writes the analysis from the outputs above so it changes when the notebook is rerun with a different date range, provider set, framework set, or symbol universe.

In [9]:
def _fmt_pct(value: float) -> str:
    return f"{value * 100:.2f}%"


def _metric_factor(metric: str) -> pd.Series:
    return factor_report.loc[factor_report["metric"] == metric].iloc[0]


best_score = comparison.sort_values("performance_score", ascending=False).iloc[0]
best_return = comparison.sort_values("total_return", ascending=False).iloc[0]
fastest = comparison.sort_values("wall_clock_seconds", ascending=True).iloc[0]
slowest = comparison.sort_values("wall_clock_seconds", ascending=False).iloc[0]
score_factor = _metric_factor("performance_score")
return_factor = _metric_factor("total_return")
speed_factor = _metric_factor("wall_clock_seconds")
max_dropped_rows = int(date_alignment_audit["dropped_rows"].max())

analysis = f"""## Analysis From This Run

- Best risk-adjusted simple score: **{best_score['provider']} + {best_score['framework']}** with performance score {_fmt_pct(float(best_score['performance_score']))}, total return {_fmt_pct(float(best_score['total_return']))}, and max drawdown {_fmt_pct(float(best_score['max_drawdown']))}.
- Best raw return: **{best_return['provider']} + {best_return['framework']}** with total return {_fmt_pct(float(best_return['total_return']))}.
- Slowest run: **{slowest['provider']} + {slowest['framework']}** at {float(slowest['wall_clock_seconds']):.2f} wall-clock seconds, so its speedup baseline is **1.00x**.
- Fastest run: **{fastest['provider']} + {fastest['framework']}** at {float(fastest['wall_clock_seconds']):.2f} wall-clock seconds, or **{float(fastest['speedup_vs_slowest']):.2f}x** faster than the slowest run.
- Date coverage is controlled with `{DATE_ALIGNMENT}`. The largest provider-specific row drop for any symbol was {max_dropped_rows} rows before backtesting.
- Price basis is controlled upstream with `{PRICE_BASIS}`. Quant Warehouse requests `splits_and_dividends` adjusted daily equity OHLC from OpenBB; this notebook does not apply notebook-side dividend adjustment.
- For performance score, the larger observed driver was **{score_factor['dominant_factor']}**: framework share {score_factor['framework_share']:.1%}, provider share {score_factor['provider_share']:.1%}, residual share {score_factor['residual_share']:.1%}.
- For raw total return, the larger observed driver was **{return_factor['dominant_factor']}**: framework share {return_factor['framework_share']:.1%}, provider share {return_factor['provider_share']:.1%}, residual share {return_factor['residual_share']:.1%}.
- For speed, the larger observed driver was **{speed_factor['dominant_factor']}**: framework share {speed_factor['framework_share']:.1%}, provider share {speed_factor['provider_share']:.1%}, residual share {speed_factor['residual_share']:.1%}.

Interpretation: after controlling date coverage and adjusted-price basis upstream in Quant Warehouse, remaining vendor differences are more likely to come from vendor OHLC/volume quality, rounding, missing bars, or corporate-action handling inside the vendor history rather than obvious adjusted-vs-unadjusted data or unequal date windows.
"""
_ = pipeline_context.update({"comparison": comparison, "symbol_detail": symbol_detail, "factor_report": factor_report, "framework_summary": framework_summary, "provider_summary": provider_summary, "runs": runs})
display(Markdown(analysis))

## Analysis From This Run

- Best risk-adjusted simple score: **yfinance + nautilus** with performance score 111.61%, total return 142.25%, and max drawdown -30.64%.
- Best raw return: **yfinance + nautilus** with total return 142.25%.
- Slowest run: **fmp + nautilus** at 5.12 wall-clock seconds, so its speedup baseline is **1.00x**.
- Fastest run: **fmp + backtesting.py** at 0.28 wall-clock seconds, or **18.50x** faster than the slowest run.
- Date coverage is controlled with `global_provider_symbol_intersection`. The largest provider-specific row drop for any symbol was 2 rows before backtesting.
- Price basis is controlled upstream with `warehouse_splits_and_dividends_adjusted_ohlc`. Quant Warehouse requests `splits_and_dividends` adjusted daily equity OHLC from OpenBB; this notebook does not apply notebook-side dividend adjustment.
- For performance score, the larger observed driver was **framework**: framework share 99.8%, provider share 0.2%, residual share 0.0%.
- For raw total return, the larger observed driver was **framework**: framework share 99.9%, provider share 0.1%, residual share 0.0%.
- For speed, the larger observed driver was **framework**: framework share 98.8%, provider share 0.4%, residual share 0.8%.

Interpretation: after controlling date coverage and adjusted-price basis upstream in Quant Warehouse, remaining vendor differences are more likely to come from vendor OHLC/volume quality, rounding, missing bars, or corporate-action handling inside the vendor history rather than obvious adjusted-vs-unadjusted data or unequal date windows.
